# Data Harmonisation

This notebook is the workflow-facing Step 2 harmonisation pipeline for the MOTEL platform. The implementation lives in `harmonise_helpers.py`; this notebook focuses on the process, the data handoff, and the outputs created during harmonisation.

> Renku note: this notebook is not intended to run in the standard Renku demo setup because the harmonisation step depends on a locally hosted LLM runtime via `ollama` and the local model `qwen3:14b`. In principle, a Renku deployment could run this workflow if it provides a compatible custom session image, enough CPU/GPU resources, and access to the local model service, but that is outside the default demo environment.

## Workflow

1. define run controls and repository paths
2. load schema context, staged unmapped data, and current registry state
3. prepare LLM guidance and optional setup controls
4. resolve entities and controlled vocabularies
5. build linked entities, write mapping files, and review the audit output


## Run Controls

Use these controls to choose whether the notebook resets staging status, backs up the database outputs, clears derived files, and limits the run to a small test slice.


In [ ]:
set_all_unmapped_to_pending = True
create_motel_db_backup = True
reset_motel_db_outputs = True
test_limit = 1   # None means no limit, otherwise set to an integer
audit_indices = [0, 1, 2]

In [2]:
import importlib

import harmonise_helpers as hh
importlib.reload(hh)

PATHS = hh.get_harmonisation_paths()

print(f"Project root: {PATHS['project_root']}")
print(f"Schema directory: {PATHS['schema_dir']}")
print(f"Staged unmapped input: {PATHS['unmapped_path']}")
print(f"Linked entity output: {PATHS['linked_entity_path']}")
print(f"Mapping directory: {PATHS['mapping_dir']}")

Project root: E:\Barton\repositories\motel-platform
Schema directory: E:\Barton\repositories\motel-platform\schema
Staged unmapped input: E:\Barton\repositories\motel-platform\motel-db\unmapped_entity\unmapped_entities_refuel.yaml
Linked entity output: E:\Barton\repositories\motel-platform\motel-db\linked_entity\linked_entity.yaml
Mapping directory: E:\Barton\repositories\motel-platform\motel-db\mapping


## Load Context

Harmonisation takes staged `unmapped_entity` records from Step 1 and resolves them into MOTEL registries, controlled vocabularies, mapping tables, and linked entities.


In [3]:
all_schemas = hh.load_all_schemas(PATHS["schema_dir"])
inputs = hh.prepare_harmonisation_inputs(
    PATHS["unmapped_path"],
    test_limit=test_limit,
    set_all_unmapped_to_pending=set_all_unmapped_to_pending,
)

all_unmapped_entities = inputs["all_unmapped_entities"]
ue = inputs["ue"]
ue_indices = inputs["ue_indices"]

print(f"Loaded schemas: {len(all_schemas)}")
print(f"Staged entities: {len(all_unmapped_entities)}")
print(f"Selected for this run: {len(ue)}")
print(f"Already mapped: {len(all_unmapped_entities) - len(ue)}")

Setting all staged entities to 'to_be_mapped' status...
Loaded schemas: 13
Staged entities: 186
Selected for this run: 186
Already mapped: 0


## Prepare Registry Context

This step uses the staged `unmapped_entity` outputs from Step 1 together with the current MOTEL registries. Workbook-derived context is expected to have been packaged during ingestion rather than loaded again here.


In [4]:
import pandas as pd

hh.pd = pd  # inject pd into harmonise_helpers module globals

all_csv_data = hh.load_all_csv_data(PATHS["database_dir"])
harmonisation_started, harmonisation_log = hh.start_harmonisation_run(
    PATHS,
    all_schemas,
    all_unmapped_entities,
    ue,
    test_limit=test_limit,
)

print(f"Loaded CSV datasets: {len(all_csv_data)}")
print(f"Harmonisation log: {harmonisation_log}")

Loaded CSV datasets: 21
Harmonisation log: ..\motel-db\log\harmonisation_20260630_181926.log


## Optional Setup Controls

Before resolving anything, the workflow can back up the current derived outputs and reset the MOTEL database outputs so the run starts from a clean state.


In [5]:
setup_actions = hh.apply_setup_controls(
    harmonisation_log,
    create_motel_db_backup=create_motel_db_backup,
    reset_motel_db_outputs=reset_motel_db_outputs,
)

print("Setup actions:", setup_actions if setup_actions else ["none"])

=== Backup saved to ..\motel-db\_backup\20260630_181926 ===
Copied:
  - ..\motel-db\controlled_vocabulary\attribute.csv
  - ..\motel-db\controlled_vocabulary\geographic_scope.csv
  - ..\motel-db\controlled_vocabulary\temporal_scope.csv
  - ..\motel-db\controlled_vocabulary\capacity_scope.csv
  - ..\motel-db\controlled_vocabulary\system_boundary.csv
  - ..\motel-db\secondary\technology.csv
  - ..\motel-db\secondary\process.csv
  - ..\motel-db\secondary\source.csv
  - ..\motel-db\controlled_vocabulary\carrier.csv
  - ..\motel-db\supplementary\contributor.csv
  - ..\motel-db\supplementary\review.csv
  - ..\motel-db\mapping/ (directory)
=== Reset complete ===
  [reset] ..\motel-db\controlled_vocabulary\attribute.csv
           columns: attribute_id, attribute_name, attribute_description, unit, data_format, ontology_iri, note
  [reset] ..\motel-db\controlled_vocabulary\geographic_scope.csv
           columns: geographic_scope, geographic_scope_description, note
  [reset] ..\motel-db\control

## Step 1: Collect Unique Candidates

The harmonisation process first deduplicates the staged names so each technology, process, source, and carrier is resolved only once during the run.


In [6]:
candidates = hh.collect_candidates(ue)

for entity_type, entity_candidates in candidates.items():
    print(f"{entity_type}: {len(entity_candidates)} unique candidates")

technology: 73 unique candidates
process: 55 unique candidates
source: 32 unique candidates
carrier: 37 unique candidates


## Step 2: Resolve Registries

This step resolves technologies, processes, sources, and carriers against the current MOTEL registries. Exact matches are reused, semantic matches can be assisted by the LLM, and genuinely new entries are created in the registries.


In [7]:
resolution = hh.resolve_entities_step(
    candidates,
    all_schemas,
    harmonisation_log=harmonisation_log,
)

resolved_ids = resolution["resolved_ids"]
resolved_names = resolution["resolved_names"]
resolution_status = resolution["resolution_status"]
resolution["counts_by_type"]


Resolving technology...
  [1/73] resolving technology: 'NH3_CCGT_El'
    -> [1/55] resolving linked process for technology 'NH3_CCGT_El': 'CCGT'
  + created: 'NH3_CCGT_El' -> 'NH3 CCGT Electrolysis'  [TECH_00001]
  [2/73] resolving technology: 'NH3_OCGT_El'
    -> [2/55] resolving linked process for technology 'NH3_OCGT_El': 'OCGT'
  + created: 'NH3_OCGT_El' -> 'NH3 Open Cycle Gas Turbine Electrolysis'  [TECH_00002]
  [3/73] resolving technology: 'CH4_CCGT_El'
  + created: 'CH4_CCGT_El' -> 'CH4 CCGT CCS'  [TECH_00003]
  [4/73] resolving technology: 'CH4_CHP_ElTh'
    -> [3/55] resolving linked process for technology 'CH4_CHP_ElTh': 'CHP plant'
  + created: 'CH4_CHP_ElTh' -> 'CH4 CHP'  [TECH_00004]
  [5/73] resolving technology: 'CH4_FuelCell_El'
    -> [4/55] resolving linked process for technology 'CH4_FuelCell_El': 'SOFC'
  + created: 'CH4_FuelCell_El' -> 'CH4 Fuel Cell El SOFC'  [TECH_00005]
  [6/73] resolving technology: 'CH4_FuelCell_ElBdg'
  [7/73] resolving technology: 'CH4_OCG

{'technology': {'exact': 0, 'llm': 4, 'created': 69},
 'process': {'exact': 0, 'llm': 0, 'created': 55},
 'source': {'exact': 1, 'llm': 0, 'created': 31},
 'carrier': {'exact': 2, 'llm': 0, 'created': 35}}

## Step 3: Resolve Controlled Vocabularies

Attributes and scope tokens are harmonised into controlled vocabularies so the linked entities can use stable foreign-key-style references instead of raw free text.


In [8]:
controlled_vocab = hh.resolve_controlled_vocabulary_step(
    all_schemas,
    ue,
    full_unmapped_path=PATHS["unmapped_path"],
    rebuild_attribute_registry=True,
    harmonisation_log=harmonisation_log,
)

attr_ids = controlled_vocab["attr_ids"]
attr_names = controlled_vocab["attr_names"]
attr_status = controlled_vocab["attr_status"]
scope_ids = controlled_vocab["scope_ids"]

Rebuilding attribute.csv from scratch...
  + attribute: 'trl' -> 'Technology Readiness Level'  [ATTR_00001]
  + attribute: 'tech_maturity' -> 'Technology Maturity'  [ATTR_00002]
  + attribute: 'technical_efficiency' -> 'Technical Efficiency'  [ATTR_00003]
  + attribute: 'theoretical_efficiency' -> 'Theoretical Efficiency'  [ATTR_00004]
  + attribute: 'lifetime_yr' -> 'Lifetime Year'  [ATTR_00005]
  + attribute: 'capex_one_time' -> 'One Time Capital Expenditure'  [ATTR_00006]
  + attribute: 'capex_per_capacity' -> 'Capital Expenditure Per Capacity'  [ATTR_00007]
  + attribute: 'opex_one_time' -> 'Operating Expenditure One Time'  [ATTR_00008]
  + attribute: 'opex_fix_pct_of_capex' -> 'Opex Fixed Percentage Of Capex'  [ATTR_00009]
  + attribute: 'opex_per_capacity_yr' -> 'Operating Expenditure Per Capacity'  [ATTR_00010]
  + attribute: 'opex_per_energy' -> 'Operating Expense Per Energy'  [ATTR_00011]
  + attribute: 'operating_temperature_c' -> 'Operating Temperature'  [ATTR_00012]
  + att

## Step 4: Build Linked Entities

Once all references are resolved, the workflow assembles linked entities by replacing raw names with resolved IDs for technologies, processes, carriers, sources, attributes, and scopes.


In [9]:
linked_output = hh.build_and_save_linked_entities(
    ue,
    ue_indices,
    all_unmapped_entities,
    PATHS["unmapped_path"],
    resolved_ids,
    attr_ids,
    attr_names,
    scope_ids,
    harmonisation_log=harmonisation_log,
)

linked_entities = linked_output["linked_entities"]
today = linked_output["today"]

linked_entities[:1]

Saved 186 new linked entities -> ../motel-db/linked_entity/linked_entity.yaml
Updated mapping status to mapped for 186 staged entities
  LE_00001  tech=TECH_00001  process=PROC_00001  scope={'geographic_scope': 'GEO_ECA', 'temporal_scope': 'TIME_2050', 'capacity_scope': '', 'system_boundary': 'BOUND_PLANT_READY_TO_OPERATE'}
  LE_00002  tech=TECH_00002  process=PROC_00002  scope={'geographic_scope': 'GEO_ECA', 'temporal_scope': 'TIME_2050', 'capacity_scope': '', 'system_boundary': 'BOUND_PLANT_READY_TO_OPERATE'}
  LE_00003  tech=TECH_00003  process=PROC_00001  scope={'geographic_scope': 'GEO_ECA', 'temporal_scope': 'TIME_2050', 'capacity_scope': 'CAP_400000', 'system_boundary': 'BOUND_PLANT_READY_TO_OPERATE'}
  LE_00004  tech=TECH_00004  process=PROC_00003  scope={'geographic_scope': 'GEO_ECA', 'temporal_scope': 'TIME_2050', 'capacity_scope': 'CAP_1000', 'system_boundary': 'BOUND_PLANT_READY_TO_OPERATE'}
  LE_00005  tech=TECH_00005  process=PROC_00004  scope={'geographic_scope': 'GEO_EC

[{'linked_entity_id': 'LE_00001',
  'tech_id': 'TECH_00001',
  'process_id': 'PROC_00001',
  'scope': {'geographic_scope': 'GEO_ECA',
   'temporal_scope': 'TIME_2050',
   'capacity_scope': '',
   'system_boundary': 'BOUND_PLANT_READY_TO_OPERATE'},
  'balancing': {'inputs': [{'carrier_id': 'CAR_00001',
     'share': 1.0,
     'unit': 'kWh'}],
   'outputs': [{'carrier_id': 'CAR_00002', 'share': 0.58, 'unit': 'kWh'}]},
  'sources': [{'source_id': 'SRC_00001',
    'linked_attribute_ids': ['ATTR_00007',
     'ATTR_00005',
     'ATTR_00010',
     'ATTR_00011',
     'ATTR_00003']},
   {'source_id': 'SRC_00002',
    'linked_attribute_ids': ['[unregistered: ratios_in]',
     '[unregistered: ratios_out]']},
   {'source_id': 'SRC_00003', 'linked_attribute_ids': ['ATTR_00012']}],
  'values': [{'attribute_id': 'ATTR_00001',
    'attribute_name': 'Technology Readiness Level',
    'value': 9,
    'time_index': 2050},
   {'attribute_id': 'ATTR_00002',
    'attribute_name': 'Technology Maturity',
    '

## Step 5: Save Mapping Files And Review Audit

The final stage writes the provenance and lookup maps, then prints a short audit report so the harmonisation decisions can be spot-checked.


In [10]:
mapping_outputs = hh.save_mapping_files_step(
    ue,
    ue_indices,
    linked_entities,
    today,
    attr_ids,
    attr_names,
    attr_status,
    scope_ids,
    harmonisation_log=harmonisation_log,
)

audit_results = hh.finish_harmonisation_run(
    harmonisation_log,
    harmonisation_started,
    ue,
    linked_entities,
    attr_ids,
    ue_indices,
    audit_indices=audit_indices,
)

Provenance map: saved 186 rows -> ..\motel-db\mapping\unmapped_to_linked.csv
Entity lookup map: attribute_map.csv  (28 rows)
Entity lookup map: geographic_scope_map.csv  (4 rows)
Entity lookup map: temporal_scope_map.csv  (4 rows)
Entity lookup map: capacity_scope_map.csv  (7 rows)
Entity lookup map: system_boundary_map.csv  (3 rows)
=== Per-Entity Audit Report ===
Auditing 3 of 186 entities.
Each entry shows the linked entity ID and how every sub-entity
(technology, process, sources, carriers, scope) was resolved.

{
  "unmapped_index": 0,
  "technology_name": "NH3_CCGT_El",
  "linked_entity_id": "LE_00001",
  "resolution": {
    "technology": {
      "original_name": "NH3_CCGT_El",
      "technology_name": "NH3 CCGT Electrolysis",
      "tech_id": "TECH_00001",
      "status": "created"
    },
    "process": {
      "original_name": "CCGT",
      "process_name": "Methane-Fueled Combined Cycle Gas Turbine with Carbon Capture and Storage (CCS) for Electricity Generation",
      "proces